# TRELLIS.2 Symmetry Projection Guidance

This notebook runs TRELLIS.2 with SymTRELLIS symmetry projection guidance. It keeps the original generation pipeline explicit: image conditioning, sparse-structure flow, sparse-structure SPG, shape-latent flow, shape-latent SPG, and mesh decoding.

Expected input image path: `examples/input.png`.

In [ ]:
import os
import sys
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "symtrellis").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import torch
import trellis2.models as models
from PIL import Image
from trellis2.modules.image_feature_extractor import DinoV3FeatureExtractor
from trellis2.modules.sparse import SparseTensor
from trellis2.pipelines.rembg import BiRefNet

from inference.trellis2 import (
    TRELLIS2_SHAPE_LATENT_CFG_INTERVAL,
    TRELLIS2_SHAPE_LATENT_CFG_RESCALE,
    TRELLIS2_SHAPE_LATENT_CFG_STRENGTH,
    TRELLIS2_SHAPE_LATENT_RESCALE_T,
    TRELLIS2_SPARSE_STRUCTURE_CFG_INTERVAL,
    TRELLIS2_SPARSE_STRUCTURE_CFG_RESCALE,
    TRELLIS2_SPARSE_STRUCTURE_CFG_STRENGTH,
    TRELLIS2_SPARSE_STRUCTURE_RESCALE_T,
    TRELLIS2FlowPredictor,
    TRELLIS2ShapeLatentNoiseSampler,
    TRELLIS2SparseLatentSymmetryProjectionNoiseSampler,
    TRELLIS2ShapeLatentView,
    TRELLIS2SparseStructureLatentNoiseSampler,
    TRELLIS2SparseStructureSymmetryProjectionNoiseSampler,
    TRELLIS2SparseStructureView,
    preprocess_image,
    trelli2_mesh_to_glb,
    trellis2_dense_grid_coords,
    trellis2_occ_to_visualization_mesh,
    trellis2_shape_latent_to_sparse_view,
    trellis2_sparse_structure_logits_to_coords,
)
from symtrellis.flow import (
    ClassifierFreeGuidanceWrapper,
    EulerSolver,
    SymmetryProjectionGuidanceWrapper,
)
from symtrellis.mapper import SymmetryProjector, from_pretrained
from symtrellis.symmetry import build_symmetry_relation_inputs, get_3d_point_group

In [ ]:
DEVICE = "cuda:0"
SIGMA_MIN = 1e-5
SEED = 42

TRELLIS2_SPARSE_STRUCTURE_STEPS = 32
TRELLIS2_SHAPE_LATENT_STEPS = 32

EXAMPLES_DIR = REPO_ROOT / "examples"
IMAGE_PATH = EXAMPLES_DIR / "input.png"
OUTPUT_GLB_PATH = EXAMPLES_DIR / "symm_enforce_trellis2.glb"

SS_FLOW_MODEL_PATH = "microsoft/TRELLIS.2-4B/ckpts/ss_flow_img_dit_1_3B_64_bf16"
SS_DECODER_PATH = "microsoft/TRELLIS-image-large/ckpts/ss_dec_conv3d_16l8_fp16"
SHAPE_FLOW_MODEL_PATH = "microsoft/TRELLIS.2-4B/ckpts/slat_flow_img2shape_dit_1_3B_512_bf16"
SHAPE_DECODER_PATH = "microsoft/TRELLIS.2-4B/ckpts/shape_dec_next_dc_f16c32_fp16"
IMAGE_COND_MODEL_NAME = "facebook/dinov3-vitl16-pretrain-lvd1689m"
REMBG_MODEL_NAME = "briaai/RMBG-2.0"

SYMTRELLIS_REPO = "symtrellis/SymTRELLIS"
SPARSE_STRUCTURE_MAPPER = "trellis2_sparse_structure_swin3d_finetune"
SHAPE_MAPPER = "trellis2_shape_swin3d_pretrain"

SHAPE_RESOLUTION = 512

SPARSE_STRUCTURE_NOISE_SYMMETRY_STRENGTH = 0.2
SPARSE_STRUCTURE_SPG_STRENGTH = 0.4
SPARSE_STRUCTURE_SPG_INTERVAL = (0.0, 0.3)

SHAPE_LATENT_NOISE_SYMMETRY_STRENGTH = 0.0
SHAPE_LATENT_SPG_STRENGTH = 0.0
SHAPE_LATENT_SPG_INTERVAL = (0.0, 0.3)

In [ ]:
ss_flow_model = models.from_pretrained(SS_FLOW_MODEL_PATH).eval()
ss_decoder = models.from_pretrained(SS_DECODER_PATH).eval()

shape_flow_model = models.from_pretrained(SHAPE_FLOW_MODEL_PATH).eval()
shape_decoder = models.from_pretrained(SHAPE_DECODER_PATH).eval()
shape_decoder.set_resolution(SHAPE_RESOLUTION)

ss_grid_size = ss_flow_model.resolution
ss_feat_dim = ss_flow_model.in_channels
shape_grid_size = shape_flow_model.resolution
shape_feat_dim = shape_flow_model.in_channels

image_cond_model = DinoV3FeatureExtractor(IMAGE_COND_MODEL_NAME, image_size=512)
rembg_model = BiRefNet(REMBG_MODEL_NAME)

In [ ]:
ss_mapper = from_pretrained(
    SPARSE_STRUCTURE_MAPPER,
    repo_id=SYMTRELLIS_REPO,
).eval()
shape_mapper = from_pretrained(
    SHAPE_MAPPER,
    repo_id=SYMTRELLIS_REPO,
).eval()

In [ ]:
ss_noise_sampler = TRELLIS2SparseStructureLatentNoiseSampler()
ss_flow_predictor = TRELLIS2FlowPredictor(model=ss_flow_model)
ss_flow_cfg_predictor = ClassifierFreeGuidanceWrapper(
    predictor=ss_flow_predictor,
    strength=TRELLIS2_SPARSE_STRUCTURE_CFG_STRENGTH,
    interval=TRELLIS2_SPARSE_STRUCTURE_CFG_INTERVAL,
    rescale=TRELLIS2_SPARSE_STRUCTURE_CFG_RESCALE,
)
ss_flow_spg_predictor = SymmetryProjectionGuidanceWrapper(
    predictor=ss_flow_cfg_predictor,
    strength=SPARSE_STRUCTURE_SPG_STRENGTH,
    interval=SPARSE_STRUCTURE_SPG_INTERVAL,
    symmetrize_target="x_start",
    rescale=0.0,
)
ss_noise_spg_sampler = TRELLIS2SparseStructureSymmetryProjectionNoiseSampler(
    sampler=ss_noise_sampler,
    symmetry_strength=SPARSE_STRUCTURE_NOISE_SYMMETRY_STRENGTH,
)

shape_noise_sampler = TRELLIS2ShapeLatentNoiseSampler()
shape_flow_predictor = TRELLIS2FlowPredictor(model=shape_flow_model)
shape_flow_cfg_predictor = ClassifierFreeGuidanceWrapper(
    predictor=shape_flow_predictor,
    strength=TRELLIS2_SHAPE_LATENT_CFG_STRENGTH,
    interval=TRELLIS2_SHAPE_LATENT_CFG_INTERVAL,
    rescale=TRELLIS2_SHAPE_LATENT_CFG_RESCALE,
)
shape_flow_spg_predictor = SymmetryProjectionGuidanceWrapper(
    predictor=shape_flow_cfg_predictor,
    strength=SHAPE_LATENT_SPG_STRENGTH,
    interval=SHAPE_LATENT_SPG_INTERVAL,
    symmetrize_target="x_start",
    rescale=0.0,
)
shape_noise_spg_sampler = TRELLIS2SparseLatentSymmetryProjectionNoiseSampler(
    sampler=shape_noise_sampler,
    symmetry_strength=SHAPE_LATENT_NOISE_SYMMETRY_STRENGTH,
)

flow_solver = EulerSolver()

In [ ]:
image = Image.open(IMAGE_PATH)

rembg_model.to(DEVICE)
processed_image = preprocess_image(image, rembg_model=rembg_model)
rembg_model.cpu()

image_cond_model.to(DEVICE)
cond = image_cond_model([processed_image])
neg_cond = torch.zeros_like(cond)
batch_size = cond.shape[0]
image_cond_model.cpu()

processed_image

In [ ]:
symmetry_label = "D5h"
symmetry_center = torch.tensor([0.0, 0.0, 0.0], device=DEVICE)
symmetry_major_axis = torch.tensor([0.0, 0.0, 1.0], device=DEVICE)
symmetry_minor_axis = torch.tensor([1.0, 0.0, 0.0], device=DEVICE)

O_dst2src, t_dst2src, s_dst2src = get_3d_point_group(
    label=symmetry_label,
    center=symmetry_center,
    major_axis=symmetry_major_axis,
    minor_axis=symmetry_minor_axis,
    include_identity=False,
)

relations = [(O_dst2src, t_dst2src, s_dst2src)]
O_dst2src.shape, t_dst2src.shape, s_dst2src.shape

In [ ]:
ss_coords = trellis2_dense_grid_coords(
    batch_size=batch_size,
    grid_size=ss_grid_size,
    device=DEVICE,
)
ss_coords_src, ss_coords_dst, ss_rows_src, ss_rows_dst, ss_O, ss_t, ss_s = build_symmetry_relation_inputs(
    coords=ss_coords,
    relations=relations,
    grid_size=ss_grid_size,
)

ss_mapper.to(DEVICE)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    ss_coeff = ss_mapper(
        coords_src=ss_coords_src,
        coords_dst=ss_coords_dst,
        O_dst2src=ss_O,
        t_dst2src=ss_t,
        s_dst2src=ss_s,
    )
ss_coeff = ss_coeff.to(device=torch.device(DEVICE), dtype=torch.float32)
ss_mapper.cpu()

ss_projector = SymmetryProjector(
    num_rows=ss_coords.shape[0],
    rows_src=ss_rows_src,
    rows_dst=ss_rows_dst,
    coeff=ss_coeff,
)

ss_coords.shape, ss_coords_src.shape, ss_coords_dst.shape

In [ ]:
ss_view = TRELLIS2SparseStructureView(
    coords=ss_coords,
    grid_size=ss_grid_size,
    batch_size=batch_size,
)

In [ ]:
torch.cuda.empty_cache()

ss_noise = ss_noise_spg_sampler.sample(
    batch_size=batch_size,
    grid_size=ss_grid_size,
    feat_dim=ss_feat_dim,
    seed=SEED,
    device=DEVICE,
    projector=ss_projector,
    to_sparse_view=ss_view.to_sparse_view,
    to_original_view=ss_view.to_original_view,
    self_include=True,
)

ss_flow_model.to(DEVICE)
ss_latent = flow_solver.sample(
    noise=ss_noise,
    predictor=ss_flow_spg_predictor,
    steps=TRELLIS2_SPARSE_STRUCTURE_STEPS,
    predictor_args={
        "cond": cond,
        "neg_cond": neg_cond,
        "projector": ss_projector,
        "to_sparse_view": ss_view.to_sparse_view,
        "to_original_view": ss_view.to_original_view,
        "self_include": True,
    },
    sigma_min=SIGMA_MIN,
    rescale_t=TRELLIS2_SPARSE_STRUCTURE_RESCALE_T,
    verbose=True,
    tqdm_desc="Sampling sparse structure",
)[-1].x_t
ss_flow_model.cpu()
torch.cuda.empty_cache()

In [ ]:
ss_decoder.to(DEVICE)
with torch.no_grad():
    ss_occ_logits = ss_decoder(ss_latent)
ss_occ = ss_occ_logits > 0
coords_symm = trellis2_sparse_structure_logits_to_coords(
    logits=ss_occ_logits,
    target_resolution=shape_grid_size,
)
ss_decoder.cpu()
torch.cuda.empty_cache()

ss_occ.shape, coords_symm.shape

In [ ]:
voxel_mesh = trellis2_occ_to_visualization_mesh(ss_occ[0, 0], y_up=True)

voxel_mesh.show()

In [ ]:
shape_coords_src, shape_coords_dst, shape_rows_src, shape_rows_dst, shape_O, shape_t, shape_s = build_symmetry_relation_inputs(
    coords=coords_symm,
    relations=relations,
    grid_size=shape_grid_size,
)

shape_mapper.to(DEVICE)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    shape_coeff = shape_mapper(
        coords_src=shape_coords_src,
        coords_dst=shape_coords_dst,
        O_dst2src=shape_O,
        t_dst2src=shape_t,
        s_dst2src=shape_s,
    )
shape_coeff = shape_coeff.to(device=torch.device(DEVICE), dtype=torch.float32)
shape_mapper.cpu()

shape_projector = SymmetryProjector(
    num_rows=coords_symm.shape[0],
    rows_src=shape_rows_src,
    rows_dst=shape_rows_dst,
    coeff=shape_coeff,
)

coords_symm.shape, shape_coords_src.shape, shape_coords_dst.shape

In [ ]:
shape_view = TRELLIS2ShapeLatentView(
    coords=coords_symm,
    sp_class=SparseTensor,
)

In [ ]:
shape_noise = shape_noise_spg_sampler.sample(
    sp_class=SparseTensor,
    coords=coords_symm,
    feat_dim=shape_feat_dim,
    grid_size=shape_grid_size,
    seed=SEED,
    device=DEVICE,
    projector=shape_projector,
    self_include=True,
)

shape_flow_model.to(DEVICE)
shape_latent = flow_solver.sample(
    noise=shape_noise,
    predictor=shape_flow_spg_predictor,
    steps=TRELLIS2_SHAPE_LATENT_STEPS,
    predictor_args={
        "cond": cond,
        "neg_cond": neg_cond,
        "projector": shape_projector,
        "to_sparse_view": shape_view.to_sparse_view,
        "to_original_view": shape_view.to_original_view,
        "self_include": True,
    },
    sigma_min=SIGMA_MIN,
    rescale_t=TRELLIS2_SHAPE_LATENT_RESCALE_T,
    verbose=True,
    tqdm_desc="Sampling shape latent",
)[-1].x_t
shape_flow_model.cpu()
torch.cuda.empty_cache()

In [ ]:
shape_latent = shape_latent.replace(
    trellis2_shape_latent_to_sparse_view(shape_latent),
)

shape_decoder.to(DEVICE)
with torch.no_grad():
    shape_mesh = shape_decoder(shape_latent)[0]
shape_decoder.cpu()
torch.cuda.empty_cache()

shape_mesh.vertices.shape, shape_mesh.faces.shape

In [ ]:
glb = trelli2_mesh_to_glb(
    shape_mesh=shape_mesh,
    res=SHAPE_RESOLUTION,
    device=torch.device(DEVICE),
    remesh=True,
    decimation_target=500000,
    remesh_project=0.9,
)

In [ ]:
glb.show()

In [ ]:
OUTPUT_GLB_PATH.parent.mkdir(exist_ok=True, parents=True)
glb.export(OUTPUT_GLB_PATH)
OUTPUT_GLB_PATH